# Explorador de FOV para cutouts MaNGA

Este notebook permite descargar y comparar una o varias galaxias con distintos campos de visión (FOV). Modifica solamente la siguiente celda de configuración y ejecuta las celdas en orden.

Un `FOV_FACTOR` menor solicita un campo más cerrado, por lo que equivale a acercarse. Sin embargo, cuando `pixscale` alcanza `PIXSCALE_MIN` (por defecto, `0.1 arcsec/pixel`), reducir más el FOV ya no aporta más resolución espacial: solo cambia el campo efectivo solicitado.

## 1. Dependencias, rutas y configuración editable

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "config.py").is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise FileNotFoundError("No encontré la raíz del repositorio (se esperaba src/config.py).")

NOTEBOOK_DIR = PROJECT_ROOT / "notebook" / "julio"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from src.config import MORFOLOGIA_CSV
from src.data.morfologia import derive_ttype_label, load_morfologia
from playground_utils import build_cutout_url, compute_pixscale, download_batch

# Cambia estos valores para probar otros objetivos y acercamientos.
TARGET_IDS = [
    "manga-10495-12704",
    "manga-7962-6102",
    "manga-8561-3704",
    "manga-8147-3704"
]
FOV_FACTORS = [8, 6, 5, 4, 3, 2, 1.5, 1, 0.75, 0.5, 0.3]  # menor factor = campo más cerrado
SIZE_PX = 256
LAYER = "ls-dr11"
BANDS = "grz"
PIXSCALE_MIN = 0.01
PIXSCALE_MAX = 0.6

EXPERIMENT = "fov_experiments"
OUTPUT_DIR = PROJECT_ROOT / "data" / "legacy_cutouts_playground" / EXPERIMENT
INDEX_CSV = OUTPUT_DIR / "fov_index.csv"
CONFIG_JSON = OUTPUT_DIR / "fov_config.json"
MAX_CONCURRENT_REQUESTS = 2
MIN_REQUEST_INTERVAL_S = 1.0
TIMEOUT_S = 60
MAX_RETRIES = 2
OVERWRITE = False

if not TARGET_IDS or not all(isinstance(name, str) and name.strip() for name in TARGET_IDS):
    raise ValueError("TARGET_IDS debe contener al menos un ID MaNGA no vacío.")
if not FOV_FACTORS or any(float(factor) <= 0 for factor in FOV_FACTORS):
    raise ValueError("FOV_FACTORS debe contener valores positivos.")
if SIZE_PX <= 0 or PIXSCALE_MIN <= 0 or PIXSCALE_MIN > PIXSCALE_MAX:
    raise ValueError("Revisa SIZE_PX y los límites de pixscale.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Salida: {OUTPUT_DIR}")
print(f"Objetivos: {TARGET_IDS}")
print(f"FOV a comparar: {FOV_FACTORS}")

## 2. Carga del catálogo y selección de galaxias

Se cargan solo los datos necesarios para construir cada cutout. Los IDs faltantes se muestran antes de hacer peticiones.

In [ ]:
REQUIRED_COLUMNS = {"name", "objra", "objdec", "PETRO_TH90"}
catalog = derive_ttype_label(load_morfologia(MORFOLOGIA_CSV))
missing_columns = sorted(REQUIRED_COLUMNS - set(catalog.columns))
if missing_columns:
    raise ValueError(f"Faltan columnas en morfologia.csv: {missing_columns}")

for column in ["objra", "objdec", "PETRO_TH90"]:
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")
catalog = catalog.dropna(subset=list(REQUIRED_COLUMNS)).copy()
catalog = catalog[catalog["PETRO_TH90"].gt(0)].drop_duplicates("name")

catalog_by_name = catalog.set_index("name")
missing_ids = [name for name in TARGET_IDS if name not in catalog_by_name.index]
selected = catalog_by_name.reindex([name for name in TARGET_IDS if name in catalog_by_name.index]).reset_index()
if selected.empty:
    raise ValueError(f"No se encontró ninguno de los IDs configurados: {TARGET_IDS}")

print(f"IDs encontrados: {len(selected)}/{len(TARGET_IDS)}")
print(f"IDs no encontrados: {missing_ids if missing_ids else 'ninguno'}")
display(selected[["name", "objra", "objdec", "PETRO_TH90", "label"]])

## 3. Escala, URL y jobs

El campo nominal es `FOV_FACTOR * PETRO_TH90`. El campo efectivo se calcula como `pixscale * SIZE_PX`, porque `pixscale` se limita entre los valores configurados.

In [ ]:
def make_fov_job(row, fov_factor):
    requested_pixscale = fov_factor * row["PETRO_TH90"] / SIZE_PX
    pixscale = compute_pixscale(
        row["PETRO_TH90"],
        fov_factor=fov_factor,
        size_px=SIZE_PX,
        pixscale_min=PIXSCALE_MIN,
        pixscale_max=PIXSCALE_MAX,
    )
    fov_tag = str(fov_factor).replace(".", "p")
    local_path = OUTPUT_DIR / row["name"] / f"fov_{fov_tag}_size_{SIZE_PX}.jpg"
    return {
        "name": row["name"],
        "label": row.get("label", "unknown"),
        "Ttype": row.get("Ttype", np.nan),
        "objra": row["objra"],
        "objdec": row["objdec"],
        "PETRO_TH90": row["PETRO_TH90"],
        "experiment": EXPERIMENT,
        "layer": LAYER,
        "bands_or_color_mode": BANDS,
        "fov_factor": float(fov_factor),
        "fov_requested_arcsec": round(float(fov_factor * row["PETRO_TH90"]), 2),
        "fov_effective_arcsec": round(float(pixscale * SIZE_PX), 2),
        "pixscale": round(float(pixscale), 4),
        "pixscale_clamped": not np.isclose(requested_pixscale, pixscale),
        "size_px": SIZE_PX,
        "url": build_cutout_url(row["objra"], row["objdec"], pixscale, SIZE_PX, LAYER, BANDS),
        "local_path": str(local_path),
    }

jobs = [make_fov_job(row, factor) for _, row in selected.iterrows() for factor in FOV_FACTORS]
job_preview = pd.DataFrame(jobs)
print(f"Jobs preparados: {len(job_preview)}")
display(job_preview[["name", "fov_factor", "fov_requested_arcsec", "fov_effective_arcsec", "pixscale", "pixscale_clamped"]])

## 4. Descarga reutilizable de cutouts RGB

Los archivos se organizan por galaxia y FOV. Con `OVERWRITE = False`, una imagen existente se reutiliza desde caché.

In [ ]:
fov_index = download_batch(
    jobs,
    max_concurrent=MAX_CONCURRENT_REQUESTS,
    min_request_interval_s=MIN_REQUEST_INTERVAL_S,
    overwrite=OVERWRITE,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    backoff_s=2.0,
    backoff_429_s=10.0,
    verbose=True,
)
fov_index["file_exists"] = fov_index["local_path"].map(lambda path: Path(path).is_file())
display(fov_index[["name", "fov_factor", "pixscale", "fov_effective_arcsec", "status", "file_exists", "error"]])

## 5. Comparación visual de acercamientos

Cada fila representa una galaxia y cada columna un valor de FOV. Si un panel tiene `pixscale_clamped=True`, el límite del visor está controlando la escala.

In [ ]:
def show_fov_comparison(index_df):
    names = index_df["name"].drop_duplicates().tolist()
    factors = list(FOV_FACTORS)
    fig, axes = plt.subplots(
        len(names), len(factors),
        figsize=(3.1 * len(factors), 3.3 * len(names)),
        squeeze=False,
    )
    for row_index, name in enumerate(names):
        galaxy_rows = index_df[index_df["name"] == name]
        for column_index, factor in enumerate(factors):
            ax = axes[row_index, column_index]
            matches = galaxy_rows[np.isclose(galaxy_rows["fov_factor"], float(factor))]
            if not matches.empty:
                record = matches.iloc[0]
                path = Path(record["local_path"])
                if record["status"] != "error" and path.is_file():
                    with Image.open(path) as image:
                        ax.imshow(image.convert("RGB"))
                    clamp = " | límite" if record["pixscale_clamped"] else ""
                    ax.set_title(
                        f"FOV={factor} | {record['pixscale']:.3f}\"/px\n"
                        f"campo={record['fov_effective_arcsec']:.1f}\"{clamp}",
                        fontsize=8,
                    )
                else:
                    ax.text(0.5, 0.5, "sin imagen", ha="center", va="center")
            ax.set_xticks([])
            ax.set_yticks([])
            if column_index == 0:
                ax.set_ylabel(name, fontsize=9)
    fig.suptitle("Comparación de FOV: menor factor = mayor acercamiento", y=1.01)
    plt.tight_layout()
    plt.show()

show_fov_comparison(fov_index)

## 6. Guardado de resultados y elección manual

El índice conserva la URL, escala y estado de cada descarga. Completa `fov_recomendado` después de revisar la cuadrícula.

In [ ]:
import json

fov_index.to_csv(INDEX_CSV, index=False)
config_to_save = {
    "target_ids": TARGET_IDS,
    "fov_factors": FOV_FACTORS,
    "size_px": SIZE_PX,
    "layer": LAYER,
    "bands": BANDS,
    "pixscale_min": PIXSCALE_MIN,
    "pixscale_max": PIXSCALE_MAX,
}
CONFIG_JSON.write_text(json.dumps(config_to_save, indent=2), encoding="utf-8")

recommendations = selected[["name", "PETRO_TH90"]].copy()
recommendations["fov_recomendado"] = pd.NA
recommendations["notas"] = ""

print(f"Índice guardado: {INDEX_CSV}")
print(f"Configuración guardada: {CONFIG_JSON}")
display(recommendations)

## 7. Prueba de FOV sin límites de `pixscale`

Esta prueba deriva `pixscale` directamente de `FOV_FACTOR * PETRO_TH90 / SIZE_PX`; no usa `PIXSCALE_MIN` ni `PIXSCALE_MAX`. Así el único parámetro que modifica el encuadre es `FOV_FACTOR`. Las imágenes se guardan en una carpeta separada para no reutilizar la caché de las pruebas acotadas.

Un valor muy pequeño de `pixscale` puede ser rechazado por el servicio o ampliar información sin añadir detalle físico nuevo; cada resultado queda registrado como `downloaded`, `cached` o `error`.

In [ ]:
FOV_FACTORS_FREE = FOV_FACTORS
FREE_PIXSCALE_OUTPUT_DIR = OUTPUT_DIR / "unclamped_pixscale"
FREE_PIXSCALE_INDEX_CSV = FREE_PIXSCALE_OUTPUT_DIR / "fov_unclamped_index.csv"


def make_unclamped_fov_job(row, fov_factor):
    pixscale = float(fov_factor * row["PETRO_TH90"] / SIZE_PX)
    fov_tag = str(fov_factor).replace(".", "p")
    pixscale_tag = f"{pixscale:.5f}".replace(".", "p")
    local_path = FREE_PIXSCALE_OUTPUT_DIR / row["name"] / f"fov_{fov_tag}_pixscale_{pixscale_tag}.jpg"
    return {
        "name": row["name"],
        "label": row.get("label", "unknown"),
        "Ttype": row.get("Ttype", np.nan),
        "objra": row["objra"],
        "objdec": row["objdec"],
        "PETRO_TH90": row["PETRO_TH90"],
        "experiment": "fov_unclamped_pixscale",
        "layer": LAYER,
        "bands_or_color_mode": BANDS,
        "fov_factor": float(fov_factor),
        "fov_requested_arcsec": round(float(fov_factor * row["PETRO_TH90"]), 4),
        "fov_effective_arcsec": round(float(pixscale * SIZE_PX), 4),
        "pixscale": round(pixscale, 6),
        "pixscale_clamped": False,
        "size_px": SIZE_PX,
        "url": build_cutout_url(row["objra"], row["objdec"], pixscale, SIZE_PX, LAYER, BANDS),
        "local_path": str(local_path),
    }


unclamped_jobs = [
    make_unclamped_fov_job(row, factor)
    for _, row in selected.iterrows()
    for factor in FOV_FACTORS_FREE
]
print("Escalas calculadas sin límites:")
display(pd.DataFrame(unclamped_jobs)[["name", "fov_factor", "pixscale", "fov_effective_arcsec", "url"]])

fov_free_index = download_batch(
    unclamped_jobs,
    max_concurrent=MAX_CONCURRENT_REQUESTS,
    min_request_interval_s=MIN_REQUEST_INTERVAL_S,
    overwrite=OVERWRITE,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    backoff_s=2.0,
    backoff_429_s=10.0,
    verbose=True,
)
fov_free_index["file_exists"] = fov_free_index["local_path"].map(lambda path: Path(path).is_file())
fov_free_index.to_csv(FREE_PIXSCALE_INDEX_CSV, index=False)

display(fov_free_index[["name", "fov_factor", "pixscale", "fov_effective_arcsec", "status", "error"]])
show_fov_comparison(fov_free_index)
print(f"Índice sin límites guardado: {FREE_PIXSCALE_INDEX_CSV}")

In [ ]:
from src.data.legacy_cutouts import (
    build_fov_jobs,
    download_fov_jobs,
    save_fov_index,
)

jobs = build_fov_jobs(
    selected,
    fov_factors=FOV_FACTORS,
    output_dir=FREE_PIXSCALE_OUTPUT_DIR,
    size_px=SIZE_PX,
    layer=LAYER,
    bands=BANDS,
    clamp_pixscale=False,
)

fov_free_index = download_fov_jobs(
    jobs,
    overwrite=OVERWRITE,
    pause_s=MIN_REQUEST_INTERVAL_S,
    max_retries=MAX_RETRIES,
    timeout_s=TIMEOUT_S,
)

fov_free_index["file_exists"] = fov_free_index["local_path"].map(
    lambda path: Path(path).is_file()
)
save_fov_index(fov_free_index, FREE_PIXSCALE_INDEX_CSV)

## 8. Cutouts artificiales desde una descarga base FOV 8

Se toma el JPEG descargado con `FOV_FACTOR_BASE = 8` y se recorta centrado para simular cada FOV de `FOV_FACTORS_FREE`. Cada recorte se reescala a `SIZE_PX` para compararlo con la descarga real de la sección 7.

La fila superior de cada comparación es la descarga real y la inferior es la simulación construida desde FOV 8. La simulación reproduce el encuadre, pero no puede recuperar el detalle que una descarga real obtiene al pedir un `pixscale` menor.

In [ ]:
FOV_FACTOR_BASE = 8.0
FOV_FACTORS_FREE = [8, 6, 5, 4, 3, 2, 1.5, 1, 0.75, 0.5, 0.3]


def artificial_cutout_from_base(image, base_fov_arcsec, target_fov_arcsec, output_size_px):
    """Recorta el centro de una imagen base y lo reescala al tamaño de salida."""
    if base_fov_arcsec <= 0 or target_fov_arcsec <= 0:
        raise ValueError("Los campos efectivos deben ser positivos.")
    if target_fov_arcsec > base_fov_arcsec:
        raise ValueError("El FOV objetivo no puede ser mayor que el FOV base.")

    source = image.convert("RGB")
    width, height = source.size
    crop_fraction = target_fov_arcsec / base_fov_arcsec
    crop_side = max(1, min(width, height, int(round(min(width, height) * crop_fraction))))

    left = (width - crop_side) // 2
    top = (height - crop_side) // 2
    crop = source.crop((left, top, left + crop_side, top + crop_side))

    return crop.resize((output_size_px, output_size_px), Image.Resampling.LANCZOS), crop_side


if fov_free_index.empty:
    raise ValueError("No hay resultados de la sección 7. Ejecútala antes de esta.")

base_rows = fov_free_index[
    np.isclose(fov_free_index["fov_factor"], FOV_FACTOR_BASE)
    & fov_free_index["file_exists"]
    & fov_free_index["status"].ne("error")
].copy()

if base_rows.empty:
    raise ValueError(
        f"No hay una descarga válida para FOV={FOV_FACTOR_BASE:g}. "
        "Incluye el factor 8 y ejecuta la sección 7."
    )

artificial_images = {}
artificial_records = []

for _, base_row in base_rows.iterrows():
    with Image.open(base_row["local_path"]) as base_image:
        for factor in FOV_FACTORS_FREE:
            target_rows = fov_free_index[
                fov_free_index["name"].eq(base_row["name"])
                & np.isclose(fov_free_index["fov_factor"], float(factor))
            ]

            if target_rows.empty:
                continue

            target_row = target_rows.iloc[0]
            simulated, crop_side = artificial_cutout_from_base(
                base_image,
                base_row["fov_effective_arcsec"],
                target_row["fov_effective_arcsec"],
                SIZE_PX,
            )

            key = (base_row["name"], float(factor))
            artificial_images[key] = simulated

            artificial_records.append({
                "name": base_row["name"],
                "fov_factor": float(factor),
                "base_fov_factor": FOV_FACTOR_BASE,
                "base_fov_effective_arcsec": base_row["fov_effective_arcsec"],
                "target_fov_effective_arcsec": target_row["fov_effective_arcsec"],
                "source_crop_side_px": crop_side,
                "output_size_px": SIZE_PX,
            })

artificial_index = pd.DataFrame(artificial_records)
display(artificial_index)

for name in base_rows["name"].tolist():
    fig, axes = plt.subplots(
        2,
        len(FOV_FACTORS_FREE),
        figsize=(3.1 * len(FOV_FACTORS_FREE), 6.2),
        squeeze=False,
    )
    galaxy_rows = fov_free_index[fov_free_index["name"].eq(name)]

    for column_index, factor in enumerate(FOV_FACTORS_FREE):
        real_ax, artificial_ax = axes[:, column_index]
        matches = galaxy_rows[
            np.isclose(galaxy_rows["fov_factor"], float(factor))
        ]
        key = (name, float(factor))

        if not matches.empty and matches.iloc[0]["file_exists"] and matches.iloc[0]["status"] != "error":
            record = matches.iloc[0]
            with Image.open(record["local_path"]) as image:
                real_ax.imshow(image.convert("RGB"))

            real_ax.set_title(
                f'FOV={factor} | {record["pixscale"]:.4f}"/px\nreal',
                fontsize=8,
            )
        else:
            real_ax.text(
                0.5, 0.5, "descarga no disponible",
                ha="center", va="center", fontsize=8,
            )

        if key in artificial_images:
            artificial_ax.imshow(artificial_images[key])
            artificial_ax.set_title("simulado desde FOV 8", fontsize=8)
        else:
            artificial_ax.text(
                0.5, 0.5, "simulación no disponible",
                ha="center", va="center", fontsize=8,
            )

        real_ax.set_xticks([])
        real_ax.set_yticks([])
        artificial_ax.set_xticks([])
        artificial_ax.set_yticks([])

        if column_index == 0:
            real_ax.set_ylabel(f"{name}\ndescarga real", fontsize=9)
            artificial_ax.set_ylabel("recorte artificial", fontsize=9)

    fig.suptitle(f"{name}: descarga real frente a recorte desde FOV 8", y=1.01)
    plt.tight_layout()
    plt.show()

# Extra: Pruebas de tamaño estandar de galaxias

In [ ]:
circle_r = [150, 125, 100, 75, 70, 60, 50, 40, 30, 20, 10, 5]

fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(15, 15))
axs = axs.flatten()

for i, r in enumerate(circle_r):
    circle = plt.Circle((0, 0), r, color='green', fill=False)
    axs[i].add_artist(circle)

    # black background
    axs[i].set_facecolor('black')

    axs[i].set_aspect('equal', 'box')
    axs[i].set_xlim(-SIZE_PX/2, SIZE_PX/2)
    axs[i].set_ylim(-SIZE_PX/2, SIZE_PX/2) 

    axs[i].set_title(f'R = {r}')
    

plt.tight_layout()
plt.show()